<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/03_advanced/01_wandb_models_registry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06. W&B Models Registry

W&B Models（実験管理・Artifact）と Weave（LLM 評価）を連携させる方法を学びます。

## 学習内容
1. `wandb.Artifact` によるモデルの保存と登録
2. W&B Registry からの取得（`run.use_artifact`）
3. Weave の `weave.Model` と W&B Artifact の橋渡し
4. 評価結果を W&B run にも記録


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0"

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import wandb
import weave
import json
import os
from pathlib import Path
from openai import OpenAI

weave_client = weave.init(f"{ENTITY}/{PROJECT}")


## 1. モデルを W&B Artifact として保存

ファインチューニング済みモデルや設定を Artifact として登録します。
ここではデモとして、プロンプト設定を含む「モデル設定ファイル」を保存します。


In [ ]:
# モデル設定を JSON として保存（実際はモデルの重みなどを保存）
model_config = {
    "model_name": "gpt-5.4-mini-2026-03-17",
    "system_message": "You are an expert grammar corrector. Return only the corrected sentence.",
    "temperature": 0.0,
    "version_notes": "Initial version with strict correction prompt",
}

config_path = Path("/tmp/model_config.json")
config_path.write_text(json.dumps(model_config, indent=2))

# W&B run を起動して Artifact を記録
run = wandb.init(
    project=PROJECT,
    entity=ENTITY,
    job_type="model-registration",
    name="register-grammar-corrector-v1",
    config=model_config,
)

artifact = wandb.Artifact(
    name="grammar-corrector",
    type="model",
    description="Grammar correction model configuration",
    metadata=model_config,
)
artifact.add_file(str(config_path), name="config.json")
run.log_artifact(artifact)
run.finish()

print(f"Artifact 登録完了: {ENTITY}/{PROJECT}/grammar-corrector:v0")
print(f"W&B UI: https://wandb.ai/{ENTITY}/{PROJECT}/artifacts/model")


## 2. Artifact の取得と Weave Model への統合

In [ ]:
class RegistryGrammarCorrector(weave.Model):
    """W&B Registry から設定を取得する Weave Model"""
    artifact_path: str
    _config: dict = {}

    def model_post_init(self, __context):
        # W&B Artifact から設定を取得
        run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            job_type="inference",
            name="load-grammar-corrector",
        )
        artifact = run.use_artifact(self.artifact_path)
        artifact_dir = artifact.download()
        config_file = Path(artifact_dir) / "config.json"
        self._config = json.loads(config_file.read_text())
        run.finish()
        print(f"モデル設定読み込み完了: {self._config['model_name']}")

    @weave.op()
    def predict(self, sentence: str) -> dict:
        oai = OpenAI()
        resp = oai.chat.completions.create(
            model=self._config["model_name"],
            messages=[
                {"role": "system", "content": self._config["system_message"]},
                {"role": "user",   "content": sentence},
            ],
            temperature=self._config["temperature"],
        )
        return {"corrected": resp.choices[0].message.content.strip()}

model = RegistryGrammarCorrector(
    artifact_path=f"{ENTITY}/{PROJECT}/grammar-corrector:latest"
)
result = model.predict("She goed to the store yesterday.")
print("修正結果:", result)


## 3. Weave Evaluation + W&B run への同時記録

In [ ]:
from weave import Evaluation, Dataset, EvaluationLogger
import pandas as pd

# 評価データセット
dataset = Dataset(
    name="grammar_registry_eval",
    rows=[
        {"sentence": "He no likes ice cream.",   "expected": "He doesn't like ice cream."},
        {"sentence": "We runned very fast.",      "expected": "We ran very fast."},
        {"sentence": "They was playing outside.", "expected": "They were playing outside."},
    ],
)
weave.publish(dataset)

@weave.op()
def exact_match(expected: str, output: dict) -> dict:
    return {"match": expected.strip() == output.get("corrected", "").strip()}

# W&B run を起動して評価結果も記録
run = wandb.init(project=PROJECT, entity=ENTITY, job_type="evaluation", name="eval-grammar-corrector-v1")

evaluation = Evaluation(name="grammar_registry_eval_v1", dataset=dataset, scorers=[exact_match])

# Weave 評価を実行（W&B run ID を属性として付与）
with weave.attributes({"wandb_run_id": run.id}):
    summary, call = await evaluation.evaluate.call(evaluation, model)

# 評価結果を W&B にも記録
metrics_flat = {}
for k, v in summary.items():
    if isinstance(v, dict):
        for kk, vv in v.items():
            metrics_flat[f"{k}/{kk}"] = vv
    else:
        metrics_flat[k] = v

run.log(metrics_flat)
run.config.update({
    "weave_eval_url": f"https://wandb.ai/{ENTITY}/{PROJECT}/r/call/{call.id}"
})
run.finish()

print("評価完了")
print("Weave 評価結果:", summary)
print(f"W&B UI: https://wandb.ai/{ENTITY}/{PROJECT}/runs")


## まとめ

次: **03_advanced/02_art_intro.ipynb** → OpenPipe ART によるエージェント RL